Here’s a complete, runnable PySpark example showing how to perform a MERGE (upsert) into a Delta Lake table in Databricks using PySpark.

The merge operation allows you to insert, update, or delete records in a Delta table based on a matching condition.

#### How It Works

**Target Table:** Existing Delta table with initial data.

**Source Data:** Contains rows to update or insert.

**merge:**

whenMatchedUpdate: Updates matching rows.

whenNotMatchedInsert: Inserts new rows.

In [0]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

In [0]:
# Sample target data (existing table)
target_data = [
    (1, "Alice", 25),
    (2, "Bob", 30),
    (3, "Charlie", 35)
]
target_df = spark.createDataFrame(target_data, ["id", "name", "age"])
target_df.write.format("delta").mode("overwrite").saveAsTable("training.default.target_data")

In [0]:
# Sample source data (new/updated records)
source_data = [
    (2, "Bob", 31),       # Update age for Bob
    (4, "David", 28)      # Insert new record
]
source_df = spark.createDataFrame(source_data, ["id", "name", "age"])

In [0]:
delta_table = DeltaTable.forName(spark,"training.default.target_data")

delta_table.alias("t").merge(source_df.alias("s"),"t.id=s.id") \
    .whenMatchedUpdate(set={"name":"s.name",
                            "age":"s.age"}) \
    .whenNotMatchedInsert(values={
        "id":"s.id",
        "name":"s.name",
        "age":"s.age"
    }) \
    .execute()